In [2]:
import os
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [3]:
documents = [
    "Ahmed is a third-year AI and Data Science student.",
    "He is learning Retrieval-Augmented Generation to build a dental lecture assistant.",
    "A convolutional neural network is commonly used for image classification tasks.",
    "Passive eruption is related to the movement of the gingival margin after tooth eruption.",
    "Power BI is used to build dashboards and visualize business data."
]

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
doc_embeddings = model.encode(documents)
print(type(doc_embeddings))
print(doc_embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [6]:
question = "What Project is Ahmed building with RAG"

question_embedding = model.encode(question)
print(question_embedding.shape)

(384,)


In [7]:
def cosine_similarity(a, b):
    dot_product = np.dot(a,b)
    mag_a = np.linalg.norm(a)
    mag_b = np.linalg.norm(b)
    return dot_product / (mag_a * mag_b)

In [8]:
similarties = []

for doc, emb in zip(documents, doc_embeddings):
    similarity = cosine_similarity(emb, question_embedding)
    similarties.append(similarity)

result = pd.DataFrame({"document": documents, "similarity": similarties})

result = result.sort_values(by="similarity", ascending=True)
result

,document,similarity
3,Passive eruption is related to the movement of...,-0.013756
2,A convolutional neural network is commonly use...,0.030239
1,He is learning Retrieval-Augmented Generation ...,0.148563
4,Power BI is used to build dashboards and visua...,0.192862
0,Ahmed is a third-year AI and Data Science stud...,0.365891


In [19]:
def retrieve_top_k(question, documents, doc_embeddings, k=3):
    question_embedding = model.encode(question)

    scores = []
    for emb in doc_embeddings:
        score = cosine_similarity(question_embedding, emb)
        scores.append(score)

    top_indices = np.argsort(scores)[::-1][:k]

    retrieved = []
    for idx in top_indices:
        retrieved.append({
            "document": documents[idx],
            "score": scores[idx]
        })

    return retrieved

In [10]:
retrieve_top_k(question, documents, doc_embeddings)

[{'document': 'Ahmed is a third-year AI and Data Science student.',
  'score': np.float32(0.36589068)},
 {'document': 'Power BI is used to build dashboards and visualize business data.',
  'score': np.float32(0.19286232)},
 {'document': 'He is learning Retrieval-Augmented Generation to build a dental lecture assistant.',
  'score': np.float32(0.14856318)}]

In [11]:
question = "What is Ahmed Trying to build"

retrieved_chunks = retrieve_top_k(question=question,
                                  documents=documents,
                                  doc_embeddings=doc_embeddings,
                                  k=3)

for item in retrieved_chunks:
    print("Score: ", round(item["score"], 4))
    print("Chunk: ", item["document"])
    print("-" * 80)

Score:  0.5202
Chunk:  Ahmed is a third-year AI and Data Science student.
--------------------------------------------------------------------------------
Score:  0.1708
Chunk:  Power BI is used to build dashboards and visualize business data.
--------------------------------------------------------------------------------
Score:  0.1658
Chunk:  He is learning Retrieval-Augmented Generation to build a dental lecture assistant.
--------------------------------------------------------------------------------


In [12]:
def split_text_into_chunks(text, chunk_size=300, overlap=50):
    text = re.sub(r"/s+", " ", text)
    
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    
    return chunks

In [13]:
lecture_text = """
Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the teeth and covers the alveolar bone. Healthy gingiva is usually firm, pink, and stippled.

A convolutional neural network is a deep learning model commonly used for image classification. It uses convolutional layers to detect features such as edges, textures, and shapes.

Power BI is a business intelligence tool used to build dashboards and analyze business data.
"""

In [14]:
chunks = split_text_into_chunks(text=lecture_text)
for i, chunk in enumerate(chunks):
    print("Chunk ", i)
    print(chunk)
    print("-"  * 80)
    

Chunk  0

Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the oppos
--------------------------------------------------------------------------------
Chunk  1
ues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the tee
--------------------------------------------------------------------------------
Chunk  2
the part of the oral mucosa that surrounds the teeth and covers the alveolar bone. Healthy gingiva is usually firm, pink, and stippled.

A convolutional neural network is a deep learning model commonly used fo

In [16]:
chunk_embeddings = model.encode(chunks)

print(chunk_embeddings.shape)

(4, 384)


In [23]:
question = "What is passive eruption?"

retrieved_chunks = retrieve_top_k(
    question=question,
    documents=chunks,
    doc_embeddings=chunk_embeddings,
    k=2
)

for item in retrieved_chunks:
    print("Score:", round(item["score"], 4))
    print("Text:", item["document"])
    print("-" * 80)


Score: 0.6646
Text: ues until the tooth reaches contact with the opposing tooth.

Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the crown of the tooth.

Gingiva is the part of the oral mucosa that surrounds the tee
--------------------------------------------------------------------------------
Score: 0.5718
Text: 
Tooth eruption is the process by which a developing tooth moves from its position in the jaw into its functional position in the oral cavity.

Active eruption refers to the movement of the tooth itself toward the occlusal plane. This movement continues until the tooth reaches contact with the oppos
--------------------------------------------------------------------------------


In [24]:
from pypdf import PdfReader

def extract_text_from_pdf(path):
    pdf = PdfReader(path)
    
    text_pages = []
    for page_number, page in enumerate(pdf.pages):
        text = page.extract_text()
        
        if text:
            text_pages.append({
                "page": page_number + 1,
                "text": text
            })
    
    return text_pages

In [25]:
pdf_path = "sample_dental_lecture.pdf"

pages = extract_text_from_pdf(pdf_path)

print("num of pages", len(pages))
print(pages[0]["page"])
print(pages[0]["text"][:1000])

num of pages 4
1
Sample Dental Lecture PDF - Page 1
 Sample Dental Lecture for RAG Testing
This PDF is made for testing a simple Retrieval-Augmented Generation pipeline. It contains clean
selectable text, so pypdf should be able to extract it without OCR.
1. Tooth Eruption
Tooth eruption is the process by which a developing tooth moves from its position inside the jaw
bone into its functional position in the oral cavity. The process involves movement through bone
and soft tissue until the tooth reaches the occlusal plane.
Active eruption refers to the actual movement of the tooth toward the occlusal plane. It continues
until the tooth contacts the opposing tooth or reaches its correct functional position.
Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It
changes the relationship between the gingiva and the anatomical crown. Clinically, passive
eruption affects how much of the crown is visible in the mouth.
2. Gingiva
Gingiva is the p

In [31]:
def chunk_pdf_pages(pages, chunk_size=800, overlap=150):
    all_chunks = []
    for page in pages:
        page_number = page["page"]
        text = page["text"]
        
        chunks = split_text_into_chunks(text=text, chunk_size=chunk_size, overlap=overlap)
        
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "page" : page_number,
                "chunk_id" : i,
                "text" : chunk
                }
            )
    return all_chunks


In [32]:
pdf_chunks = chunk_pdf_pages(
    pages,
    chunk_size=800,
    overlap=150
)

print("Number of chunks:", len(pdf_chunks))

print(pdf_chunks[0]["page"])
print(pdf_chunks[0]["chunk_id"])
print(pdf_chunks[0]["text"][:500])

Number of chunks: 9
1
0
Sample Dental Lecture PDF - Page 1
 Sample Dental Lecture for RAG Testing
This PDF is made for testing a simple Retrieval-Augmented Generation pipeline. It contains clean
selectable text, so pypdf should be able to extract it without OCR.
1. Tooth Eruption
Tooth eruption is the process by which a developing tooth moves from its position inside the jaw
bone into its functional position in the oral cavity. The process involves movement through bone
and soft tissue until the tooth reaches the occlu


In [33]:
pdf_text = [chunk["text"] for chunk in pdf_chunks]

pdf_embeddings = model.encode(pdf_text)

print(pdf_embeddings.shape)

(9, 384)


In [36]:
def retrieve_top_k_pdf(question, pdf_chunks, pdf_embeddings, k=3):
    question_embedding = model.encode(question)
    
    scores = []
    for emb in pdf_embeddings:
        score = cosine_similarity(emb, question_embedding)
        scores.append(score)
    
    top_indices = np.argsort(scores)[::-1][:k]
    
    retrieved = []
    for idx in top_indices:
        retrieved.append({
            "page": pdf_chunks[idx]["page"],
            "chunk_id": pdf_chunks[idx]["chunk_id"],
            "text": pdf_chunks[idx]["text"],
            "score": scores[idx]
            }
        )
    return retrieved

In [38]:
question = "What are the reasons for process suspension?"

retrieved = retrieve_top_k_pdf(
    question=question,
    pdf_chunks=pdf_chunks,
    pdf_embeddings=pdf_embeddings,
    k=3
)

for item in retrieved:
    print("Page:", item["page"])
    print("Chunk:", item["chunk_id"])
    print("Score:", round(item["score"], 4))
    print(item["text"][:1000])
    print("-" * 100)

Page: 4
Chunk: 0
Score: 0.1575
Sample Dental Lecture PDF - Page 4
 Exam-Style Summary
7. Important Exam Points
Active eruption is movement of the tooth itself toward the occlusal plane. Passive eruption is
related to apical movement of the gingival margin after eruption.
Reversible pulpitis usually causes short pain that stops after removing the stimulus. Irreversible
pulpitis usually causes lingering or spontaneous pain.
RAG does not train a new language model. Instead, it retrieves relevant document chunks and
uses them as context for the model response.
8. Example Questions
Question 1: What is passive eruption?
Question 2: What is the difference between reversible and irreversible pulpitis?
Question 3: Why does a RAG system split documents into chunks?
Question 4: What is transfer learning in image classification?
En
----------------------------------------------------------------------------------------------------
Page: 2
Chunk: 0
Score: 0.142
Sample Dental Lecture PDF - Page 2
 D

In [41]:
import faiss

embedding_dim = pdf_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

normalized_embeddings = pdf_embeddings.copy()
faiss.normalize_L2(normalized_embeddings)

index.add(normalized_embeddings)

print("vectors in index: ", index.ntotal)

vectors in index:  9


In [42]:
def retrieve_faiss(question, pdf_chunks, index, k=3):
    question_embedding = model.encode([question])
    question_embedding = np.array(question_embedding).astype("float32")
    
    faiss.normalize_L2(question_embedding)
    
    scores, indices = index.search(question_embedding, k)
    
    retrieved = []
    
    for idx, score in zip(indices[0], scores[0]):
        retrieved.append({
                    "page": pdf_chunks[idx]["page"],
                    "chunk_id": pdf_chunks[idx]["chunk_id"],
                    "text": pdf_chunks[idx]["text"],
                    "score": float(score)
                })
    
    return retrieved
    

In [43]:
question = "Explain passive eruptions"

retrieved = retrieve_faiss(question=question, pdf_chunks=pdf_chunks, index=index, k=3)

for item in retrieved:
    print("Page:", item["page"])
    print("Score:", round(item["score"], 4))
    print(item["text"][:1000])
    print("-" * 100)

Page: 1
Score: 0.5501
oth or reaches its correct functional position.
Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It
changes the relationship between the gingiva and the anatomical crown. Clinically, passive
eruption affects how much of the crown is visible in the mouth.
2. Gingiva
Gingiva is the part of the oral mucosa that surrounds the teeth and covers the alveolar bone.
Healthy gingiva is usually firm, pink, and stippled. The gingival margin forms the border of the
gingiva around the tooth.
The attached gingiva is firmly bound to the underlying periosteum and tooth. The free gingiva
surrounds the tooth but is not attached directly to the tooth surface.

----------------------------------------------------------------------------------------------------
Page: 1
Score: 0.5053
Sample Dental Lecture PDF - Page 1
 Sample Dental Lecture for RAG Testing
This PDF is made for testing a simple Retrieval-Augmented Generation pipeline. It

In [44]:
def build_prompt(question, retrieved):
    context_parts = []
    
    for idx, chunk in enumerate(retrieved, start=1):
        context_parts.append(f"[Source {idx} | Page {chunk['page']}]\n{chunk['text']}")
        
    context = '/n/n'.join(context_parts)
    
    prompt = f"""
You are a helpful study assistant.

Answer the question using ONLY the provided context.

If the answer is not found in the context, say:
"I could not find the answer in the provided document."

Question:
{question}

Context:
{context}

Answer:
"""

    return prompt

In [45]:
question = "Explain passive eruptions"

retrieved = retrieve_faiss(question=question, pdf_chunks=pdf_chunks, index=index, k=3)

prompt = build_prompt(question=question, retrieved=retrieved)

print(prompt)


You are a helpful study assistant.

Answer the question using ONLY the provided context.

If the answer is not found in the context, say:
"I could not find the answer in the provided document."

Question:
Explain passive eruptions

Context:
[Source 1 | Page 1]
oth or reaches its correct functional position.
Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It
changes the relationship between the gingiva and the anatomical crown. Clinically, passive
eruption affects how much of the crown is visible in the mouth.
2. Gingiva
Gingiva is the part of the oral mucosa that surrounds the teeth and covers the alveolar bone.
Healthy gingiva is usually firm, pink, and stippled. The gingival margin forms the border of the
gingiva around the tooth.
The attached gingiva is firmly bound to the underlying periosteum and tooth. The free gingiva
surrounds the tooth but is not attached directly to the tooth surface.
/n/n[Source 2 | Page 1]
Sample Dental L

In [48]:
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [51]:
def generate_answer(prompt):
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents=prompt
    )
    
    return response

In [54]:
def answer_question(question, pdf_chunks, index, k=3):
    retrieved = retrieve_faiss(
        question=question,
        pdf_chunks=pdf_chunks,
        index=index,
        k=k
    )

    prompt = build_prompt(question, retrieved)

    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "sources": retrieved
    }

In [55]:
result = answer_question(
    question="What is passive eruption?",
    pdf_chunks=pdf_chunks,
    index=index,
    k=3
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(f"Page {source['page']} | Score {source['score']:.4f}")
    print(source["text"][:500])
    print("-" * 80)

Question:
What is passive eruption?

Answer:
sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='Passive eruption refers to the apical migration of the gingival margin after the tooth has erupted. It changes the relationship between the gingiva and the anatomical crown and affects how much of the crown is visible in the mouth.'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='3YsHaphcu_vE3w_Y_trJBQ' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=43,
  prompt_token_count=587,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=587
    ),
  ],
  thoughts_token_count=175,
  total_token_count=805
) model_status=None automatic_function_calling_history=[] parsed=None

Sources:
Page 1 | Sc